# Tabular ML — dataset.csv
18-step record of exactly what this pipeline run did to your data, in the same step order as the generic reference notebook -- every cell below is real, runnable code, built from this run's actual recorded values.

**Target column:** `exam_score`  |  **Task:** regression  |  **Best model:** `gradient_boosting`

## 1.1 Data Upload

Tries THREE sources, in order: (1) the ORIGINAL raw file this run was pointed at (`dataset.csv`) -- real `pd.read_csv`, live data, matching the reference notebook's own first step exactly; (2) **`notebook_data_sample.csv`**, a genuine sample of this run's own raw data that ships in this SAME output folder specifically so this notebook is self-contained -- steps 1.2-1.18 all run for real off this, no external path needed; (3) only if neither file is reachable, the 5-row head/tail snapshot already recorded in `pipeline_run_summary.json`, clearly labeled as a reconstruction, so steps 1.2-1.6 never silently pretend a 5-row preview is the full dataset.

In [ ]:
import json
import os
import pandas as pd
import numpy as np

with open('pipeline_run_summary.json', 'r', encoding='utf-8') as f:
    summary = json.load(f)
eda = summary.get('eda', {})

INPUT_PATH = 'dataset.csv'
TARGET_COL = 'exam_score'
SAMPLE_PATH = 'notebook_data_sample.csv'
_RAW_DATA_AVAILABLE = False
if INPUT_PATH and os.path.exists(INPUT_PATH):
    df = pd.read_csv(INPUT_PATH)
    _RAW_DATA_AVAILABLE = True
    print(f"Loaded LIVE data: {df.shape[0]} rows x {df.shape[1]} columns from '{INPUT_PATH}'")
elif os.path.exists(SAMPLE_PATH):
    df = pd.read_csv(SAMPLE_PATH)
    _RAW_DATA_AVAILABLE = True
    print(f"Original path not reachable here -- loaded this run's own REAL data sample: "
          f"{df.shape[0]} rows x {df.shape[1]} columns from '{SAMPLE_PATH}' "
          f"(ships in this same folder, so every step below still runs for real).")
else:
    print(f"Neither '{INPUT_PATH}' nor '{SAMPLE_PATH}' found from this notebook's "
          f"working directory -- falling back to this run's recorded 5-row EDA "
          f"snapshot for steps 1.2-1.6. Steps 1.7-1.16 (which need a real dataframe) "
          f"will be skipped; steps 1.17-1.18 still run fully, they only need files "
          f"in this same folder.")
    df = pd.DataFrame(eda.get('head_sample', []))
print(f"Live data available: {_RAW_DATA_AVAILABLE}")

## 1.2 Check Head Value

In [ ]:
df.head()

## 1.3 Check Tail Value

In [ ]:
df.tail() if _RAW_DATA_AVAILABLE else pd.DataFrame(eda.get('tail_sample', []))

## 1.4 Check Missing Values

Computed LIVE via `df.isnull().sum()` when the raw data loaded; falls back to this run's recorded missing-value report otherwise.

In [ ]:
if _RAW_DATA_AVAILABLE:
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
else:
    missing_df = pd.DataFrame(eda.get('missing_report', {})).T
missing_df.sort_values('missing_count', ascending=False) if not missing_df.empty else print('No missing values recorded for this run.')

## 1.5 Check Info

In [ ]:
df.info() if _RAW_DATA_AVAILABLE else pd.Series(eda.get('dtype_report', {}), name='dtype')

## 1.6 Check Uniques

In [ ]:
(df.nunique() if _RAW_DATA_AVAILABLE else pd.Series(eda.get('nunique_report', {}), name='n_unique')).sort_values(ascending=False)

## 1.7 Handle Missing / Leaking Columns

Drops this run's own recorded ID/leak-pattern columns (`drop_leaking_columns()` -- word-boundary name matching, e.g. `customer_id`/`index`), then splits the remaining columns into numeric/categorical -- the two lists step 1.10 below builds the real `ColumnTransformer` from. Needs the live dataframe from 1.1; skipped otherwise.

In [ ]:
if _RAW_DATA_AVAILABLE:
    dropped_leak_cols = ['student_id']
    df = df.drop(columns=[c for c in dropped_leak_cols if c in df.columns])
    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET_COL]
    categorical_cols = [c for c in df.select_dtypes(exclude=[np.number]).columns if c != TARGET_COL]
    print(f'Dropped leak/ID column(s): {dropped_leak_cols}')
    print(f'{len(numeric_cols)} numeric, {len(categorical_cols)} categorical feature column(s) remain.')
else:
    print('Skipped -- needs the live dataframe from step 1.1.')

## 1.8 Exploratory Data Analysis

Live Pearson correlation matrix + distribution histograms over this run's real numeric features.

In [ ]:
if _RAW_DATA_AVAILABLE and len(numeric_cols) >= 2:
    import matplotlib.pyplot as plt

    corr = df[numeric_cols].corr()
    fig, ax = plt.subplots(figsize=(max(6, 0.5*len(numeric_cols)), max(5, 0.5*len(numeric_cols))))
    im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
    ax.set_xticks(range(len(numeric_cols))); ax.set_xticklabels(numeric_cols, rotation=90)
    ax.set_yticks(range(len(numeric_cols))); ax.set_yticklabels(numeric_cols)
    fig.colorbar(im, ax=ax, label='Pearson correlation')
    ax.set_title('Feature Correlation Matrix (this run, live)'); plt.tight_layout(); plt.show()

    df[numeric_cols].hist(figsize=(12, 8), bins=30)
    plt.tight_layout(); plt.show()
else:
    print('Skipped -- needs the live dataframe (step 1.1) with >= 2 numeric columns.')

## 1.9 Feature Engineering

Live IQR-based outlier clipping (clip, not drop -- keeps row count/labels intact) + skew-based `log1p` on heavily right-skewed positive columns, both computed from THIS data's actual quantiles/skew, not a canned rule.

In [ ]:
if _RAW_DATA_AVAILABLE:
    for col in numeric_cols:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        df[col] = df[col].clip(lower, upper)

    for col in numeric_cols:
        if (df[col] > 0).all() and df[col].skew() > 1.0:
            df[col] = np.log1p(df[col])
            print(f"Log-transformed '{col}' (skew was {df[col].skew():.2f} pre-transform)")
else:
    print('Skipped -- needs the live dataframe from step 1.1.')

## 1.10 Categorical Encoding

Builds the real, leak-free `sklearn.pipeline.Pipeline` + `ColumnTransformer` -- median-impute then scale for numeric columns, most-frequent-impute then one-hot encode for categorical columns. Constructed here, but **not fit yet** -- it's fit for the first time in step 1.13, on the train split only, so no validation/test row ever influences an imputation median, a scaler's mean/std, or which one-hot categories exist.

In [ ]:
if _RAW_DATA_AVAILABLE:
    from sklearn.pipeline import Pipeline
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder

    _SCALERS = {'standard': StandardScaler, 'minmax': MinMaxScaler, 'robust': RobustScaler}
    SCALE_METHOD = 'standard'
    scaler_cls = _SCALERS.get(SCALE_METHOD, StandardScaler)

    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', scaler_cls()),
    ])
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ])
    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline, numeric_cols),
        ('cat', categorical_pipeline, categorical_cols),
    ])
    print(f'ColumnTransformer built: {len(numeric_cols)} numeric (impute+{SCALE_METHOD} scale), '
          f'{len(categorical_cols)} categorical (impute+one-hot) column(s). Not fit yet.')
else:
    print('Skipped -- needs the live dataframe from step 1.1.')

## 1.11 Feature Selection

Drops numeric columns that are highly correlated (> 0.90 Pearson) with another feature already kept -- computed live, on the real correlation matrix, not a stored list.

In [ ]:
if _RAW_DATA_AVAILABLE and len(numeric_cols) >= 2:
    CORR_THRESHOLD = 0.90
    corr_matrix = df[numeric_cols].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]
    numeric_cols = [c for c in numeric_cols if c not in to_drop]
    print(f'Dropped {len(to_drop)} highly-correlated (> {CORR_THRESHOLD}) redundant feature(s): {to_drop}')
else:
    print('Skipped -- needs the live dataframe (step 1.1) with >= 2 numeric columns.')

## 1.12 Data Splitting

Real `train_test_split`, 70/15/15, stratified for classification -- same split shape this run itself used.

In [ ]:
if _RAW_DATA_AVAILABLE:
    from sklearn.model_selection import train_test_split

    IS_CLASSIFICATION = False
    X = df[numeric_cols + categorical_cols]
    y = df[TARGET_COL]
    strat = y if IS_CLASSIFICATION else None
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=strat)
    strat_temp = y_temp if IS_CLASSIFICATION else None
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=42, stratify=strat_temp)
    print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')
else:
    IS_CLASSIFICATION = False
    print('Skipped -- needs the live dataframe from step 1.1.')

## 1.13 Feature Scaling

**`preprocessor.fit_transform(X_train)`** -- the `ColumnTransformer` built in 1.10 is fit here, for the first time, on the TRAIN split only; `X_val`/`X_test` only ever see `.transform()` (never `.fit()`), which is what actually makes this leak-free.

In [ ]:
if _RAW_DATA_AVAILABLE:
    X_train_t = preprocessor.fit_transform(X_train)
    X_val_t = preprocessor.transform(X_val)
    X_test_t = preprocessor.transform(X_test)
    print(f'Transformed shapes -- train: {X_train_t.shape}, val: {X_val_t.shape}, test: {X_test_t.shape}')
else:
    print('Skipped -- needs the live dataframe from step 1.1.')

## 1.14 Model Build — Baseline

This run's own CV comparison included a linear baseline: **`ridge`** (CV score: 0.6665). Genuinely fit here on the real, leak-free-transformed train split.

In [ ]:
if _RAW_DATA_AVAILABLE:
    from sklearn.linear_model import LogisticRegression, LinearRegression

    baseline_model = (LogisticRegression(max_iter=1000, random_state=42) if IS_CLASSIFICATION
                       else LinearRegression())
    baseline_model.fit(X_train_t, y_train)
    print(f'Baseline ({type(baseline_model).__name__}) trained on {X_train_t.shape[0]} rows.')
else:
    print('Skipped -- needs the live dataframe from step 1.1.')

## 1.15 Advanced Model Training

This run's actual best-performing model family, **`gradient_boosting`**, from this pipeline's own model catalog -- genuinely instantiated and fit here (not a hardcoded stand-in family). Every model this run's cross-validation actually compared, ranked by CV score:

In [ ]:
if _RAW_DATA_AVAILABLE:
    from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor

    best_model_obj = GradientBoostingClassifier(random_state=42) if IS_CLASSIFICATION else GradientBoostingRegressor(random_state=42)
    best_model_obj.fit(X_train_t, y_train)
    print(f'{type(best_model_obj).__name__} trained on {X_train_t.shape[0]} rows.')

    import matplotlib.pyplot as plt
    cv_comparison = [{'model_name': 'gradient_boosting', 'cv_folds': 5, 'scoring': 'r2', 'fold_scores': [0.6963841491870737, 0.686200392409783, 0.6821957728654386, 0.7038490246084199, 0.6903774525405311], 'mean_score': 0.6918013583222493, 'std_score': 0.007639552785989008, 'elapsed_sec': 13.82930040359497, 'sample_weighted': True}, {'model_name': 'lightgbm', 'cv_folds': 5, 'scoring': 'r2', 'fold_scores': [0.6790259457983575, 0.6802390451098261, 0.6690899680254849, 0.6933061365660567, 0.6814577287714296], 'mean_score': 0.680623764854231, 'std_score': 0.007710610252839207, 'elapsed_sec': 2.742436408996582, 'sample_weighted': True}, {'model_name': 'linear_regression', 'cv_folds': 5, 'scoring': 'r2', 'fold_scores': [0.6749242762489185, 0.6642659387989107, 0.6586875088372837, 0.6722419444302963, 0.6620960551556774], 'mean_score': 0.6664431446942173, 'std_score': 0.006153746360377449, 'elapsed_sec': 0.08035659790039062, 'sample_weighted': True}, {'model_name': 'random_forest', 'cv_folds': 5, 'scoring': 'r2', 'fold_scores': [0.6669695248393669, 0.6611194884861465, 0.6593722950766133, 0.6783600387501517, 0.6661037124888421], 'mean_score': 0.6663850119282241, 'std_score': 0.006644185230164931, 'elapsed_sec': 49.462849855422974, 'sample_weighted': True}, {'model_name': 'svm', 'cv_folds': 5, 'scoring': 'r2', 'fold_scores': [0.6514641138798657, 0.6535526476287741, 0.6451726017668576, 0.6667170213578739, 0.6540314306559678], 'mean_score': 0.6541875630578678, 'std_score': 0.007015718391215377, 'elapsed_sec': 61.99350595474243, 'sample_weighted': True}, {'model_name': 'knn', 'cv_folds': 5, 'scoring': 'r2', 'fold_scores': [0.5795236436550244, 0.5767880573182473, 0.5769195513758857, 0.5849645268875884, 0.5845752014321322], 'mean_score': 0.5805541961337756, 'std_score': 0.0035798319366356778, 'elapsed_sec': 1.0355188846588135, 'sample_weighted': False}, {'model_name': 'decision_tree', 'cv_folds': 5, 'scoring': 'r2', 'fold_scores': [0.3181383601534796, 0.358344235059743, 0.31084360171930503, 0.3287581407019837, 0.31367784553984757], 'mean_score': 0.32595243663487183, 'std_score': 0.017303773292681807, 'elapsed_sec': 0.6438524723052979, 'sample_weighted': True}, {'model_name': 'lasso', 'cv_folds': 5, 'scoring': 'r2', 'fold_scores': [-0.16026715656891155, -0.16765605685803475, -0.14271119443026148, -0.16080928135353822, -0.141953158101352], 'mean_score': -0.1546793694624196, 'std_score': 0.010415251024443063, 'elapsed_sec': 0.08446049690246582, 'sample_weighted': True}, {'model_name': 'elastic_net', 'cv_folds': 5, 'scoring': 'r2', 'fold_scores': [-0.16026715656891155, -0.16765605685803475, -0.14271119443026148, -0.16080928135353822, -0.141953158101352], 'mean_score': -0.1546793694624196, 'std_score': 0.010415251024443063, 'elapsed_sec': 0.044267892837524414, 'sample_weighted': True}, {'model_name': 'xgboost', 'cv_folds': 5, 'scoring': 'r2', 'fold_scores': [-157810.872633401, -158168.25843639788, -157936.11971272674, -161827.60750596775, -155720.72258107964], 'mean_score': -158292.71617391458, 'std_score': 1974.1039351109541, 'elapsed_sec': 1.2514123916625977, 'sample_weighted': True}]
    cv_df = pd.DataFrame(cv_comparison)
    if not cv_df.empty:
        display_cols = [c for c in ['model_name', 'mean_score', 'std_score'] if c in cv_df.columns]
        cv_df = cv_df[display_cols].sort_values('mean_score', ascending=False)
        display(cv_df)
        plt.figure(figsize=(8, 4))
        plt.barh(cv_df['model_name'][::-1], cv_df['mean_score'][::-1], color='steelblue')
        plt.title('Model comparison (this run)'); plt.xlabel('CV score'); plt.tight_layout(); plt.show()
    else:
        print('No CV comparison recorded for this run.')
else:
    print('Skipped -- needs the live dataframe from step 1.1.')

## 1.16 Hyperparameter Tuning

**Method:** `randomized`  |  **Best CV score found:** 0.7036  |  **Best params:** `{'n_estimators': 200, 'max_depth': 2, 'learning_rate': 0.1}` -- re-applied and refit below.

In [ ]:
if _RAW_DATA_AVAILABLE:
    hyperparam_search = {'model_name': 'gradient_boosting', 'method': 'randomized', 'scoring': 'r2', 'best_score': 0.7036335278229513, 'best_params': {'n_estimators': 200, 'max_depth': 2, 'learning_rate': 0.1}, 'elapsed_sec': 436.0630986690521, 'sample_weighted': True}
    if hyperparam_search.get('best_params'):
        try:
            best_model_obj.set_params(**hyperparam_search['best_params'])
            best_model_obj.fit(X_train_t, y_train)
            print(f"Refit with recorded best params: {hyperparam_search['best_params']}")
        except (ValueError, TypeError) as e:
            print(f'Could not re-apply recorded best params to this model instance '
                  f'({e}) -- keeping step 1.15\'s model as-is.')
    else:
        print('No hyperparameter search recorded for this run -- keeping step 1.15\'s model.')
else:
    print('Skipped -- needs the live dataframe from step 1.1.')

## 1.17 Evaluation Metrics

Loads **`validation_report.csv`** -- this run's own recorded true/predicted values for the model that was actually deployed (self-contained; ships in this same output folder, works even when 1.1-1.16 above were skipped). Renders a real Confusion Matrix + ROC-AUC curve for classification, or Residual + Prediction Error plots for regression (this run's target was log1p-transformed for TRAINING only -- the pipeline itself already `expm1`-inverts every prediction back to the original scale before writing `validation_report.csv`, so the values loaded below are already on the original scale and must NOT be inverse-transformed again here).

In [ ]:
import pandas as pd
import numpy as np
vr = pd.read_csv('validation_report.csv')
# NOTE: this run log1p-transformed the target for TRAINING only -- the pipeline
# already expm1-inverted every prediction back to the original scale before saving
# validation_report.csv, so true_label/predicted_label below are ALREADY on the
# original scale. Re-applying expm1 here would double-invert them and produce
# nonsense metrics -- these are used as-is, with NO further transform.
y_true = vr['true_label'].values
y_pred = vr['predicted_label'].values
proba_cols = [c for c in vr.columns if c.startswith('proba_')]
print(f'Loaded {len(vr)} real test-set row(s) from validation_report.csv.')

In [ ]:
IS_CLASSIFICATION = False
import matplotlib.pyplot as plt

if IS_CLASSIFICATION:
    from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                                  roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)

    print(f'Accuracy:  {accuracy_score(y_true, y_pred):.4f}')
    print(f'Precision: {precision_score(y_true, y_pred, average="weighted", zero_division=0):.4f}')
    print(f'Recall:    {recall_score(y_true, y_pred, average="weighted", zero_division=0):.4f}')
    print(f'F1-Score:  {f1_score(y_true, y_pred, average="weighted", zero_division=0):.4f}')

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm).plot(ax=ax, cmap='Blues', colorbar=True)
    ax.set_title('Confusion Matrix (real test-set predictions, this run)')
    plt.tight_layout(); plt.show()

    if proba_cols:
        classes = sorted(pd.unique(y_true))
        if len(classes) == 2 and len(proba_cols) >= 2:
            pos_col = proba_cols[-1]
            y_score = vr[pos_col].values
            y_true_bin = (y_true == classes[-1]).astype(int)
            fpr, tpr, _ = roc_curve(y_true_bin, y_score)
            auc = roc_auc_score(y_true_bin, y_score)
            plt.figure(figsize=(6, 5))
            plt.plot(fpr, tpr, label=f'ROC curve (AUC = {auc:.3f})', color='darkorange')
            plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance')
            plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
            plt.title('ROC-AUC Curve (real test-set scores, this run)'); plt.legend()
            plt.tight_layout(); plt.show()
        else:
            print(f'{len(classes)}-class problem -- one-vs-rest ROC-AUC (macro): '
                  f'{roc_auc_score(pd.get_dummies(y_true), vr[proba_cols].values, multi_class="ovr"):.4f}')
    else:
        print('No per-class probabilities were recorded for this run (this model has no '
              'predict_proba, or probability capture failed) -- confusion matrix above is '
              'real, but no ROC-AUC curve can be drawn without continuous scores.')
else:
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    y_true_f = y_true.astype(float); y_pred_f = y_pred.astype(float)
    rmse = mean_squared_error(y_true_f, y_pred_f) ** 0.5
    print(f'RMSE: {rmse:.4f}')
    print(f'MAE:  {mean_absolute_error(y_true_f, y_pred_f):.4f}')
    print(f'R2:   {r2_score(y_true_f, y_pred_f):.4f}')

    residuals = y_true_f - y_pred_f
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(y_pred_f, residuals, alpha=0.5, edgecolor='k', linewidth=0.3)
    axes[0].axhline(0, color='red', linestyle='--')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual (true - predicted)')
    axes[0].set_title('Residual Plot (real test-set predictions, this run)')

    axes[1].scatter(y_true_f, y_pred_f, alpha=0.5, edgecolor='k', linewidth=0.3)
    lims = [min(y_true_f.min(), y_pred_f.min()), max(y_true_f.max(), y_pred_f.max())]
    axes[1].plot(lims, lims, color='red', linestyle='--', label='Perfect prediction')
    axes[1].set_xlabel('True'); axes[1].set_ylabel('Predicted')
    axes[1].set_title('Prediction Error Plot'); axes[1].legend()
    plt.tight_layout(); plt.show()

## 1.18 Save Model

`tabular_model.onnx`/`tabular_model.json` (model), `feature_recipe.json` (the exact encode/scale recipe served by `app.py`), `validation_report.csv` (per-row test predictions) -- all in this same output folder. This pipeline never uses pickle/joblib for model artifacts, so the cell below verifies the actual exported ONNX file still produces the SAME prediction recorded in `validation_report.csv`, when `onnxruntime` is installed. The exported model was itself fit on the log1p-transformed target, so its raw ONNX output is still in log-space -- `np.expm1` is applied to it below before comparing, the same inverse-transform the pipeline itself applies before writing `predicted_label`; comparing them without that would always show a spurious mismatch even on a perfectly correct export.

In [ ]:
import os

onnx_candidates = [f for f in os.listdir('.') if f.endswith('.onnx')]
json_candidates = [f for f in os.listdir('.') if f == 'tabular_model.json']
print(f'Exported model file(s) found in this folder: {onnx_candidates + json_candidates}')

if onnx_candidates:
    try:
        import onnxruntime as ort
        sess = ort.InferenceSession(onnx_candidates[0])
        sample = vr.drop(columns=['true_label', 'predicted_label'] + proba_cols).iloc[[0]]
        input_name = sess.get_inputs()[0].name
        onnx_out = sess.run(None, {input_name: sample.values.astype(np.float32)})[0]
        # This model was fit on log1p(y_train) -- its raw output is still in
        # log-space; expm1 it back to the original scale before comparing, the
        # same inverse-transform the pipeline applies before writing predicted_label.
        onnx_out = np.expm1(np.asarray(onnx_out, dtype=np.float64))
        print(f'ONNX sanity check -- row 0: recorded prediction={vr["predicted_label"].iloc[0]}, '
              f'ONNX output={onnx_out}')
    except ImportError:
        print('onnxruntime not installed -- skipping the load-and-verify sanity check. '
              'Install with: pip install onnxruntime==1.19.2')
    except Exception as e:
        print(f'ONNX sanity check failed ({type(e).__name__}: {e}) -- the exported file '
              f'may need the exact feature order feature_recipe.json records.')
else:
    print('No .onnx file in this folder (linear model -> tabular_model.json instead, '
          'or export was skipped this run) -- nothing to sanity-check here.')

### Deployment bundle: app.py + requirements.txt + Dockerfile + k8s.yaml + ci.yml

Builds the serving app for THIS model only (no code for any other model type) plus the container/CI files, from the model artifacts in the output folder. Needs the pipeline `.py` next to this notebook (or set `AUTOML_PIPELINE_PY`).

In [ ]:
import glob, importlib.util, os, sys

OUTPUT_DIR = os.environ.get("AUTOML_OUTPUT_DIR", ".")   # folder holding the exported model artifacts (onnx / json / adapter ...)
MODEL_TYPE = 'TABULAR_ML'   # None -> auto-detected from pipeline_run_summary.json / the artifacts present
# Only used when the model artifacts are NOT in OUTPUT_DIR yet: the pipeline then trains its own version of this model
# on DATA_PATH and builds the bundle. Reuses INPUT_PATH / DOCS_DIR / TARGET_COL from earlier cells when they exist.
DATA_PATH = os.environ.get("AUTOML_DATA_PATH") or globals().get("INPUT_PATH") or globals().get("DOCS_DIR") or ""
TRAIN_FLAGS = {"target_col": globals().get("TARGET_COL")}   # NLP/LLM: {"text_col": ..., "label_col": ...}; RNN: {"sequence_mode": ..., "sequence_col": ...}

_here = os.getcwd()
_candidates = [os.environ.get("AUTOML_PIPELINE_PY", "")]
for _d in (_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))):
    _candidates += sorted(glob.glob(os.path.join(_d, "*universal_automl_pipeline*.py")), reverse=True)
PIPELINE_PY = next((c for c in _candidates if c and os.path.isfile(c)), None)
if PIPELINE_PY is None:
    raise FileNotFoundError("Pipeline .py not found - copy it next to this notebook or set AUTOML_PIPELINE_PY.")

_spec = importlib.util.spec_from_file_location("automl_pipeline", PIPELINE_PY)
automl_pipeline = importlib.util.module_from_spec(_spec)
sys.modules["automl_pipeline"] = automl_pipeline
_spec.loader.exec_module(automl_pipeline)

deploy = automl_pipeline.rebuild_deployment_bundle(OUTPUT_DIR, model_type=MODEL_TYPE,
                                                   data_path=DATA_PATH or None, train_flags=TRAIN_FLAGS)
print("model_type        :", deploy["model_type"], "(detected via", deploy["detected_via"] + ")")
print("app.py            :", deploy["app_py"])
print("missing artifacts :", deploy["missing_artifacts"] or "none")
if deploy["api_key"]:
    print("API key (shown once):", deploy["api_key"])